# Datathon 2026 Round 1 - MCQ Solver


In [14]:
import os

csv_files = sorted([f for f in os.listdir('.') if f.endswith('.csv')])
print(f"Found {len(csv_files)} csv files:")
for f in csv_files:
    print('-', f)

Found 14 csv files:
- customers.csv
- geography.csv
- inventory.csv
- order_items.csv
- orders.csv
- payments.csv
- products.csv
- promotions.csv
- returns.csv
- reviews.csv
- sales.csv
- sample_submission.csv
- shipments.csv
- web_traffic.csv


In [ ]:
import pandas as pd

required_files = [
    'products.csv',
    'customers.csv',
    'geography.csv',
    'orders.csv',
    'order_items.csv',
    'payments.csv',
    'returns.csv',
    'web_traffic.csv',
]

missing_files = [f for f in required_files if not os.path.exists(f)]
if missing_files:
    raise FileNotFoundError('Thieu cac file sau: ' + ', '.join(missing_files))

products = pd.read_csv('products.csv')
customers = pd.read_csv('customers.csv')
geography = pd.read_csv('geography.csv')
orders = pd.read_csv('orders.csv')
order_items = pd.read_csv(
    'order_items.csv',
    dtype={'promo_id': 'string', 'promo_id_2': 'string'},
    low_memory=False,
)
payments = pd.read_csv('payments.csv')
returns = pd.read_csv('returns.csv')
web_traffic = pd.read_csv('web_traffic.csv')

orders['order_date'] = pd.to_datetime(orders['order_date'])

print('Loaded successfully')
print('orders shape:', orders.shape)
print('order_items shape:', order_items.shape)
print('returns shape:', returns.shape)

Loaded successfully
orders shape: (646945, 8)
order_items shape: (714669, 7)
returns shape: (39939, 7)


## Q1. Median inter-order gap

In [ ]:
q1_df = orders.sort_values(['customer_id', 'order_date'])[['customer_id', 'order_date']].copy()
q1_df['prev_order_date'] = q1_df.groupby('customer_id')['order_date'].shift()
q1_df['gap_days'] = (q1_df['order_date'] - q1_df['prev_order_date']).dt.days

q1_value = q1_df['gap_days'].dropna().median()
print('Q1 median inter-order gap:', q1_value)
print('Answer: C) 144 ngay')

Q1 median inter-order gap: 144.0
Answer: C) 144 ngay


## Q2. Segment co gross margin trung binh cao nhat

In [ ]:
q2_table = products.copy()
q2_table['gross_margin'] = (q2_table['price'] - q2_table['cogs']) / q2_table['price']
q2_table = q2_table.groupby('segment', as_index=False)['gross_margin'].mean().sort_values('gross_margin', ascending=False)

display(q2_table)
print('Answer:', q2_table.iloc[0]['segment'])
print('MCQ choice: D) Standard')

,segment,gross_margin
6,Standard,0.313442
5,Premium,0.285377
1,All-weather,0.284176
0,Activewear,0.265600
4,Performance,0.263650
2,Balanced,0.258038
7,Trendy,0.240758
3,Everyday,0.236343


Answer: Standard
MCQ choice: D) Standard


## Q3. Ly do tra hang pho bien nhat trong Streetwear

In [ ]:
q3_table = (
    returns.merge(products[['product_id', 'category']], on='product_id', how='left')
    .query("category == 'Streetwear'")['return_reason']
    .value_counts()
)

display(q3_table)
print('Answer:', q3_table.index[0])
print('MCQ choice: B) wrong_size')

,count
return_reason,
wrong_size,7626
defective,4330
not_as_described,3854
changed_mind,3830
late_delivery,2159


Answer: wrong_size
MCQ choice: B) wrong_size


## Q4. Traffic source co bounce rate trung binh thap nhat

In [ ]:
q4_table = web_traffic.groupby('traffic_source', as_index=False)['bounce_rate'].mean().sort_values('bounce_rate')

display(q4_table)
print('Answer:', q4_table.iloc[0]['traffic_source'])
print('MCQ choice: C) email_campaign')

,traffic_source,bounce_rate
1,email_campaign,0.004458
5,social_media,0.004476
3,paid_search,0.004478
4,referral,0.004499
2,organic_search,0.004504
0,direct,0.004511


Answer: email_campaign
MCQ choice: C) email_campaign


## Q5. Ty le dong order_items co promo_id khac null

In [ ]:
q5_value = order_items['promo_id'].notna().mean() * 100
print(f'Q5 promo usage ratio: {q5_value:.4f}%')
print('MCQ choice: C) 39%')

Q5 promo usage ratio: 38.6635%
MCQ choice: C) 39%


## Q6. Age group co so don trung binh moi khach hang cao nhat

In [ ]:
customer_orders = orders.groupby('customer_id').size().rename('orders_count')

q6_table = (
    customers[['customer_id', 'age_group']]
    .merge(customer_orders, on='customer_id', how='left')
    .assign(orders_count=lambda df: df['orders_count'].fillna(0))
    .loc[lambda df: df['age_group'].notna()]
    .groupby('age_group', as_index=False)['orders_count']
    .mean()
    .sort_values('orders_count', ascending=False)
)

display(q6_table)
print('Answer:', q6_table.iloc[0]['age_group'])
print('MCQ choice: A) 55+')

,age_group,orders_count
4,55+,5.406851
3,45-54,5.357241
2,35-44,5.337343
1,25-34,5.245226
0,18-24,5.226656


Answer: 55+
MCQ choice: A) 55+


## Q7. Region tao ra tong revenue cao nhat

In [ ]:
q7_table = (
    order_items.merge(orders[['order_id', 'zip']], on='order_id', how='left')
    .merge(geography[['zip', 'region']], on='zip', how='left')
    .assign(revenue=lambda df: df['quantity'] * df['unit_price'])
    .groupby('region', as_index=False)['revenue']
    .sum()
    .sort_values('revenue', ascending=False)
)

display(q7_table)
print('Answer:', q7_table.iloc[0]['region'])
print('MCQ choice: C) East')

,region,revenue
1,East,7.637533e+09
0,Central,4.941908e+09
2,West,3.851035e+09


Answer: East
MCQ choice: C) East


## Q8. Payment method pho bien nhat trong cancelled orders

In [ ]:
q8_table = orders.loc[orders['order_status'] == 'cancelled', 'payment_method'].value_counts()

display(q8_table)
print('Answer:', q8_table.index[0])
print('MCQ choice: A) credit_card')

,count
payment_method,
credit_card,28452
cod,15468
paypal,7817
apple_pay,5190
bank_transfer,2535


Answer: credit_card
MCQ choice: A) credit_card


## Q9. Size co return rate cao nhat

In [ ]:
size_lines = (
    order_items.merge(products[['product_id', 'size']], on='product_id', how='left')
    .groupby('size')
    .size()
    .rename('order_item_lines')
)

size_returns = (
    returns.merge(products[['product_id', 'size']], on='product_id', how='left')
    .groupby('size')
    .size()
    .rename('return_records')
)

q9_table = pd.concat([size_lines, size_returns], axis=1).fillna(0)
q9_table['return_rate'] = q9_table['return_records'] / q9_table['order_item_lines']
q9_table = q9_table.sort_values('return_rate', ascending=False)

display(q9_table)
print('Answer:', q9_table.index[0])
print('MCQ choice: A) S')

,order_item_lines,return_records,return_rate
size,,,
S,172042,9723,0.056515
L,173174,9741,0.056250
M,176428,9820,0.055660
XL,193025,10655,0.055200


Answer: S
MCQ choice: A) S


## Q10. Installments co payment_value trung binh cao nhat

In [ ]:
q10_table = payments.groupby('installments', as_index=False)['payment_value'].mean().sort_values('payment_value', ascending=False)

display(q10_table)
print('Answer:', int(q10_table.iloc[0]['installments']))
print('MCQ choice: C) 6 ky')

,installments,payment_value
3,6,24446.654403
2,3,24399.635486
4,12,24245.772694
0,1,24113.274166
1,2,708.473729


Answer: 6
MCQ choice: C) 6 ky


## Tong hop dap an

In [ ]:
summary = pd.DataFrame(
    {
        'Question': ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8', 'Q9', 'Q10'],
        'Choice': ['C', 'D', 'B', 'C', 'C', 'A', 'C', 'A', 'A', 'C'],
        'Answer': [
            '144 ngay',
            'Standard',
            'wrong_size',
            'email_campaign',
            '~39%',
            '55+',
            'East',
            'credit_card',
            'S',
            '6 ky',
        ],
    }
)

display(summary)

,Question,Choice,Answer
0,Q1,C,144 ngay
1,Q2,D,Standard
2,Q3,B,wrong_size
3,Q4,C,email_campaign
4,Q5,C,~39%
5,Q6,A,55+
6,Q7,C,East
7,Q8,A,credit_card
8,Q9,A,S
9,Q10,C,6 ky
